In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.1 MB/s eta 0:00:00


In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.1/112.6 GB disk)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11l.pt")

In [ ]:
class_list = model.names
print(class_list)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [ ]:
import cv2
cap = cv2.VideoCapture("/content/drive/MyDrive/VehicleCounting/Vehicle Dataset Sample 3.mp4")

In [ ]:
from collections import defaultdict
class_counts = defaultdict(int)

In [ ]:
cross_ids= set()

In [ ]:
output_video = "/content/drive/MyDrive/VehicleCounting/output.mp4"
fps = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

In [ ]:
from google.colab.patches import cv2_imshow
import time
frame_count=0
while cap.isOpened():
  ret,frame = cap.read()
  if not ret:
    break


  results = model.track(frame, persist = True, classes=[1,2,3,5,6,7])
  if results[0].boxes.data is not None:
    boxes = results[0].boxes.xyxy.cpu()
    track_ids = results[0].boxes.id.int().cpu().tolist()
    class_indexes =results[0].boxes.cls.int().cpu().tolist()


    cv2.line(frame, (690-320-220, 250+430), (1130+330+300, 250+430), (0, 0, 255), 3)
    #430 chilo aage

    for box,track_id,class_index in zip(boxes,track_ids,class_indexes):
      x1,y1,x2,y2= map(int,box)
      c1 = int((x1 + x2) / 2)
      c2 = int((y1 + y2) / 2)
      class_name = class_list[class_index]


      cv2.circle(frame, (c1,c2), 4, (0,0,255), -1)

      cv2.putText(frame, f"ID: {track_id} {class_name}",
            (x1,y1-10), cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (0,255,255), 2)

      cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 2)

      if (c1 >= 690-320-220 and c1 <= 1130+330+300) and c2 > 250+430 and track_id not in cross_ids:
        # if((c1 >= 690-320-220 and c1 <= 1130+330+300) and c2 > 250+430)
        cross_ids.add(track_id)
        class_counts[class_name]+=1

    y_first=70
    for class_name, count in class_counts.items():
      cv2.putText(frame, f"{class_name}: {count}", (0,y_first), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2)
      y_first+=100


    if frame_count < 30:
      print("Showing frame:", frame_count)
      cv2_imshow(frame)
      time.sleep(0.05)
    # if frame_count >= 5:
    #   break


  out.write(frame)
  frame_count += 1
cap.release()
# cv2.destroyAllWindows()
out.release()